# Install Package

In [ ]:
# define the train and test split random seed
SEED = 42
# install kaggle
!pip install kaggle

# Data Preparation for Toxic Text

In [ ]:
import os
import json

# where to store your kaggle.json,according to your Kaggle Environment
kaggle_dir = '/root/.config/kaggle'
os.makedirs(kaggle_dir, exist_ok=True)

#write your Kaggle API key to kaggle.json
kaggle_api = {
    "username": "geronimohe",
    "key": "6fbd8ff41e453aa983090b2e8001419b"
}
kaggle_file = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_file, "w") as f:
    json.dump(kaggle_api, f)

# download dataset for toxic text
base_dir = '/data/ningyuanhe/EvasionFromICL'
data_dir = os.path.join(base_dir, "data")
os.makedirs(data_dir, exist_ok=True)
toxic_text_dir = os.path.join(data_dir, "toxic_text")
target_path = toxic_text_dir

!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -p {target_path}

In [ ]:
# unzip the downloaded dataset to toxic_text_dir
import zipfile
download_path = os.path.join(toxic_text_dir,"jigsaw-toxic-comment-classification-challenge.zip")
extract_path = toxic_text_dir

with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

download_path = os.path.join(toxic_text_dir,"train.csv.zip")
extract_path = toxic_text_dir
with zipfile.ZipFile(download_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# preprocess the data
# We only use the train dataset rather than the test dataset. This is because the test datset has no labels
# In our experiments, we use column "label" to denote the label name, column "text" to specify the sample text
raw_data_dir = os.path.join(toxic_text_dir,"train.csv")
df = pd.read_csv(raw_data_dir)
columns_to_drop = ['id','severe_toxic','obscene','threat','insult','identity_hate']
df.drop(columns=columns_to_drop, inplace=True)
df = df.rename(columns={'comment_text': 'text'})
df.insert(0, 'idx', range(0, len(df) ))
df['toxic'] = df['toxic'].apply(lambda x: 1 if x == 0 else 0)
df = df.rename(columns={'toxic': 'label_value'})
df.insert(3, 'label','')
df['label'] = df['label_value'].map({1:'benign' , 0: 'toxic'})

# store the normalized toxic text data
toxic_text_data_dir = os.path.join(toxic_text_dir,"data")
os.makedirs(toxic_text_data_dir, exist_ok=True)
normalized_toxic_text_data_dir = os.path.join(toxic_text_data_dir,"normalized_toxic_text_data.csv")
df.to_csv(normalized_toxic_text_data_dir, index=False)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#split the data according to label
df = pd.read_csv(normalized_toxic_text_data_dir)
if 'label' in df.columns:
    toxic_data = df[df['label'] == 'toxic']
    benign_data = df[df['label'] == 'benign']

    toxic_data_dir = os.path.join(toxic_text_data_dir,"toxic_data_csv")
    toxic_data.to_csv(toxic_data_dir, index=False)

    benign_data_dir = os.path.join(toxic_text_data_dir,"benign_data_csv")
    benign_data.to_csv(benign_data_dir, index=False)

    print("successfully split data into toxic_data.csv and benign_data.csv")
else:
    print("the 'label' column don't exist in CSV")


#sample 2500 from the toxic_data and benign_data to reduce the size of the dataset
df = pd.read_csv(toxic_data_dir)
sampled_toxic_data = df.sample(n=2500, random_state=SEED) 
sampled_toxic_data_dir = os.path.join(toxic_text_data_dir,"sampled_toxic_data.csv")
sampled_toxic_data.to_csv(sampled_toxic_data_dir, index=False)

df = pd.read_csv(benign_data_dir)
sampled_benign_data = df.sample(n=2500, random_state=SEED)  
sampled_benign_data_dir = os.path.join(toxic_text_data_dir,"sampled_benign_data.csv")
sampled_benign_data.to_csv(sampled_benign_data_dir, index=False)


#split the data into 80%train and 20%test data
df = pd.read_csv(sampled_toxic_data_dir)
toxic_train_df, toxic_test_df = train_test_split(df, test_size=0.2, random_state=SEED)
toxic_train_dir = os.path.join(toxic_text_data_dir,"toxic_trian.csv")
toxic_train_df.to_csv(toxic_train_dir, index=False)
toxic_test_dir = os.path.join(toxic_text_data_dir,"toxic_test.csv")
toxic_test_df.to_csv(toxic_test_dir, index=False)

df = pd.read_csv(sampled_benign_data_dir)
benign_train_df, benign_test_df = train_test_split(df, test_size=0.2, random_state=SEED)
benign_train_dir = os.path.join(toxic_text_data_dir,"benign_trian.csv")
benign_train_df.to_csv(benign_train_dir, index=False)
benign_test_dir = os.path.join(toxic_text_data_dir,"benign_test.csv")
toxic_test_df.to_csv(benign_test_dir, index=False)


#combine the toxic and benign train data into final train data
#combine the toxic and benign test data into final test data
toxic_train = pd.read_csv(toxic_train_dir)
benign_train = pd.read_csv(benign_train_dir)
train_data = pd.concat([toxic_train, benign_train], ignore_index=True)
train_data_dir = os.path.join(toxic_text_data_dir,"train.csv")
train_data.to_csv(train_data_dir, index=False)

toxic_test = pd.read_csv(toxic_test_dir)
benign_test = pd.read_csv(benign_test_dir)
test_data = pd.concat([toxic_test, benign_test], ignore_index=True)
test_data_dir = os.path.join(toxic_text_data_dir,"test.csv")
test_data.to_csv(test_data_dir, index=False)

successfully split data into toxic_data.csv and benign_data.csv


# Data Preparation for illicit Promotion

In [ ]:
# preprocess the data
# In our experiments, we use column "label" to denote the label name, column "text" to specify the sample text
import pandas as pd
from sklearn.model_selection import train_test_split
illicit_promotion_dir = os.path.join(data_dir,"illicit_promotion")
os.makedirs(illicit_promotion_dir,exist_ok=True)

df = pd.read_csv(os.path.join(illicit_promotion_dir,"balanced_binary_data.csv"))


df.insert(0, 'idx', range(0, len(df) ))
df.drop(columns=['source'], inplace=True)

df['label_value'] = df['label'].map({'benign': 1, 'illicit': 0})

df.insert(df.columns.get_loc('label'), 'label_value', df.pop('label_value'))
normalized_data_dir = os.path.join(illicit_promotion_dir,"normalized_binary_data.csv")
df.to_csv(normalized_data_dir, index=False)  

#We split the data into 80% train data and 20% test data
df = pd.read_csv(normalized_data_dir)  

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df.to_csv(os.path.join(illicit_promotion_dir,"train.csv"), index=False)  
test_df.to_csv(os.path.join(illicit_promotion_dir,"test.csv"), index=False) 
   